# Olist E-Commerce Analysis - Data Cleaning

Now that I know what the data looks like, I need to clean it up before doing any real analysis. A few things I ran into:
- all the date columns are stored as strings - need to parse them to actual datetime
- there are about 3k orders that were cancelled or never delivered, and they don't have delivery dates, so I'm dropping those
- product category names are in Portuguese, need to join the translation table to get English names
- at the end I'm building one big joined table so I don't have to keep re-merging everything in every notebook

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_DIR = os.path.abspath("..")
RAW_PATH = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_DIR, "data", "processed")
CHARTS_PATH = os.path.join(BASE_DIR, "outputs", "charts")

print("paths set up")

paths set up


In [3]:
# reloading everything - can't assume they're in memory from nb1
customers          = pd.read_csv(os.path.join(RAW_PATH, "olist_customers_dataset.csv"))
orders             = pd.read_csv(os.path.join(RAW_PATH, "olist_orders_dataset.csv"))
order_items        = pd.read_csv(os.path.join(RAW_PATH, "olist_order_items_dataset.csv"))
order_payments     = pd.read_csv(os.path.join(RAW_PATH, "olist_order_payments_dataset.csv"))
order_reviews      = pd.read_csv(os.path.join(RAW_PATH, "olist_order_reviews_dataset.csv"))
products           = pd.read_csv(os.path.join(RAW_PATH, "olist_products_dataset.csv"))
sellers            = pd.read_csv(os.path.join(RAW_PATH, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_PATH, "product_category_name_translation.csv"))

print("done.")

done.


## Step 1 - fixing data types and parsing dates

In [4]:
# these are all strings right now, need to convert them to actual dates
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("date columns fixed")
print(orders.dtypes[date_cols])

date columns fixed
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Step 2 - handling cancelled and invalid orders

In [5]:
# checking what statuses we have before filtering
print("order status breakdown:")
print(orders['order_status'].value_counts())

order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
# keeping only delivered orders for the main analysis
# cancelled/unavailable ones don't have delivery dates so they'd mess up the numbers
print("before filtering:", len(orders))
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()  # using copy() to avoid SettingWithCopyWarning
print("after filtering (delivered only):", len(orders_delivered))
print(f"removed {len(orders) - len(orders_delivered)} non-delivered orders")

before filtering: 99441
after filtering (delivered only): 96478
removed 2963 non-delivered orders


## Step 3 - translating product categories

In [7]:
# the category names are in Portuguese, need to join the translation table
products_translated = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

# check how many didn't get a translation
missing = products_translated['product_category_name_english'].isnull().sum()
print(f"{missing} products couldn't be translated - filling with 'unknown'")
products_translated['product_category_name_english'] = (
    products_translated['product_category_name_english'].fillna('unknown')
)
print("done.")

623 products couldn't be translated - filling with 'unknown'
done.


## Step 4 - building the master table

In [8]:
# joining everything together into one big table
# starting with delivered orders, then adding items, payments, reviews, customers

master = orders_delivered.merge(order_items, on='order_id', how='left')
master = master.merge(order_payments, on='order_id', how='left')
master = master.merge(
    order_reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)
master = master.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
    on='customer_id',
    how='left'
)
master = master.merge(
    products_translated[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)
master = master.merge(
    sellers[['seller_id', 'seller_state', 'seller_city']],
    on='seller_id',
    how='left'
)

print(f"master table shape: {master.shape}")
master.head()

master table shape: (115723, 25)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_type,payment_installments,payment_value,review_score,customer_unique_id,customer_state,customer_city,product_category_name_english,seller_state,seller_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,credit_card,1.0,18.12,4.0,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,housewares,SP,maua
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,voucher,1.0,2.00,4.0,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,housewares,SP,maua
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,voucher,1.0,18.59,4.0,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,housewares,SP,maua
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,...,boleto,1.0,141.46,4.0,af07308b275d755c9edb36a90c618231,BA,barreiras,perfumery,SP,belo horizonte
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,...,credit_card,3.0,179.12,5.0,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis,auto,SP,guariba


In [9]:
# some orders have multiple payment rows (split payments)
# keeping only the first payment record per order to avoid double-counting revenue
# this took me a while to figure out - had inflated row counts before catching this
master = master.drop_duplicates(subset=['order_id', 'order_item_id'])
print(f"after deduplication: {master.shape}")

after deduplication: (110197, 25)

In [10]:
# final null check on the master table
nulls = master.isnull().sum()
nulls = nulls[nulls > 0]
print("remaining nulls in master table:")
print(nulls)

remaining nulls in master table:
order_approved_at                 15
order_delivered_carrier_date       2
order_delivered_customer_date      8
payment_sequential                 3
payment_type                       3
payment_installments               3
payment_value                      3
review_score                     827
dtype: int64


In [11]:
# saving the clean master table so I don't have to redo all this every time
master.to_csv(os.path.join(PROCESSED_PATH, "master_clean.csv"), index=False)
print("saved to data/processed/master_clean.csv")

saved to data/processed/master_clean.csv


Master table is built and saved. It's got all the core info in one place - orders, items, payments, reviews, customers and sellers joined together. From here I can get into the actual analysis without worrying about the data plumbing.